In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

In [2]:
# =========================================================================
# 1. تحميل البيانات الأصلية
# =========================================================================
print("--- 1. Loading Data ---")
df_production = pd.read_csv('L_P.csv')
df_downtime = pd.read_csv('L_D.csv')
df_factor = pd.read_csv('D_F.csv')
df_products = pd.read_csv('Product.csv')

--- 1. Loading Data ---


In [3]:
# =========================================================================
# 2. تجهيز البيانات للموديل الأول (توقع حالة الإنتاج Delayed أو On Time)
# =========================================================================
print("\n--- 2. Preparing Features for Production Status ---")


--- 2. Preparing Features for Production Status ---


In [4]:
# تحويل التواريخ والأوقات
df_production['Date'] = pd.to_datetime(df_production['Date'])
df_production['Month'] = df_production['Date'].dt.month
df_production['Day'] = df_production['Date'].dt.day

In [5]:
df_production['Start Time'] = pd.to_datetime(df_production['Start Time'], format='%I:%M:%S %p')
df_production['Start_Hour'] = df_production['Start Time'].dt.hour

In [6]:
# دمج وقت الإنتاج القياسي من ملف المنتجات (مهم جداً كـ Feature)
df_production = df_production.merge(df_products[['Product_id', 'Min batch time']], on='Product_id', how='left')

In [7]:
# اختيار الـ Features (شيلنا وقت النهاية والمدة الفعلية منعاً للغش والتسريب)
X_status = df_production[['Product_id', 'Operator_Name', 'Shift', 'Month', 'Day', 'Start_Hour', 'Min batch time']].copy()
y_status = df_production['Production_status']

In [8]:
# تحويل النصوص إلى أرقام
le = LabelEncoder()
for col in ['Product_id', 'Operator_Name', 'Shift']:
    X_status[col] = le.fit_transform(X_status[col])

In [9]:
# =========================================================================
# 3. تدريب وتقييم الموديل الأول (Classification)
# =========================================================================
print("\n--- 3. Training Production Status Model (Random Forest) ---")

loo = LeaveOneOut()
model_status = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# التقييم الذكي سطر سطر لتجنب الصدفة في حجم الداتا الصغير
y_pred_cv = cross_val_predict(model_status, X_status, y_status, cv=loo)

print("\n[التقرير النهائي لحالة الإنتاج]:")
print(classification_report(y_status, y_pred_cv))

# حفظ التوقعات للموديل الأول في ملف مستقل
df_production['Production_status_prediction'] = y_pred_cv
df_production.to_csv('Production_status_prediction.csv', index=False)
print("Saved: Production_status_prediction.csv")


--- 3. Training Production Status Model (Random Forest) ---

[التقرير النهائي لحالة الإنتاج]:
              precision    recall  f1-score   support

     Delayed       0.92      0.94      0.93        35
     On Time       0.00      0.00      0.00         3

    accuracy                           0.87        38
   macro avg       0.46      0.47      0.46        38
weighted avg       0.84      0.87      0.86        38

Saved: Production_status_prediction.csv


In [10]:
# --- الجزء الثاني: Down Time Prediction (ربط المشغل والوردية) ---
print("\n--- Training Downtime Regressor Model ---")


--- Training Downtime Regressor Model ---


In [11]:
# =========================================================================
# 4. تجهيز وتدريب الموديل الثاني (توقع مدة الأعطال بالدقائق)
# =========================================================================
print("\n--- 4. Training Downtime Regressor Model ---")

# دمج الجداول للحصول على صورة العطل كاملة (نوع العطل، الخطأ، المشغل، الوردية)
dt_merged = df_downtime.merge(df_factor[['Factor_ID', 'Operator Error']], left_on='Fact_id', right_on='Factor_ID', how='left')
dt_merged['Operator Error'] = dt_merged['Operator Error'].map({'Yes': 1, 'No': 0})

# ربط الـ Batch ببيانات الإنتاج الأصلية للحصول على المشغل والوردية قبل الحذف
dt_merged = dt_merged.merge(df_production[['Batch', 'Product_id', 'Operator_Name', 'Shift', 'Min batch time']], on='Batch', how='left')

# اختيار الـ Features للموديل التاني
dt_features = dt_merged[['Fact_id', 'Operator Error', 'Product_id', 'Operator_Name', 'Shift', 'Min batch time']].copy()
y_downtime = dt_merged['DownTime_MIn']

# تحويل النصوص لأرقام
for col in ['Product_id', 'Operator_Name', 'Shift']:
    dt_features[col] = le.fit_transform(dt_features[col].astype(str))

# تقسيم الداتا لـ Train و Test (الداتا هنا 61 سطر فـ التقسيم العادي شغال)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(dt_features, y_downtime, test_size=0.2, random_state=42)

# عمل Scaling (تنسيق للأرقام) لأن الموديلات الخطية بتحتاجه
scaler = StandardScaler()
X_train_d_scaled = scaler.fit_transform(X_train_d)
X_test_d_scaled = scaler.transform(X_test_d)

# استخدام Ridge Regression لأنه يحمي الموديل من الـ Overfitting في الداتا الصغيرة
model_downtime = Ridge(alpha=1.0)
model_downtime.fit(X_train_d_scaled, y_train_d)

# التوقع والتقييم
y_pred_d = model_downtime.predict(X_test_d_scaled)

print("\n[التقرير النهائي لتوقع وقت الأعطال]:")
print(f'Mean Squared Error (MSE): {mean_squared_error(y_test_d, y_pred_d):.2f}')
print(f'R-squared (R2 Score): {r2_score(y_test_d, y_pred_d):.2f}')

# حفظ توقعات الأعطال للداتا كلها في ملف مستقل
X_all_scaled = scaler.transform(dt_features)
dt_merged['DownTime_MIn_prediction'] = model_downtime.predict(X_all_scaled)
dt_merged.to_csv('Downtime_prediction.csv', index=False)
print("Saved: Downtime_prediction.csv")


--- 4. Training Downtime Regressor Model ---

[التقرير النهائي لتوقع وقت الأعطال]:
Mean Squared Error (MSE): 122.71
R-squared (R2 Score): 0.29
Saved: Downtime_prediction.csv


In [ ]:
# Pseudocode for the Prescriptive Recommendation System
FUNCTION generate_mitigation_strategy(product_id, assigned_operator, target_shift, predicted_status, predicted_downtime):
    INITIALIZE recommendations list
    
    IF predicted_status EQUALS "Delayed":
        # 1. Identify alternative optimal operators for this specific product
        best_operators = query_dataframe(
            filter_by(Product_id == product_id, Production_status == "On Time")
            .group_by(Operator_Name)
            .count_descending()
        )
        
        # 2. Extract top recommendation
        IF best_operators IS NOT EMPTY:
            top_operator = best_operators[0].Operator_Name
            APPEND "Recommendation 1: Consider reassigning this batch to Operator [" + top_operator + "] to minimize delay risks." TO recommendations
        
        # 3. Analyze shift alternatives
        best_shifts = query_dataframe(
            filter_by(Product_id == product_id, Production_status == "On Time")
            .group_by(Shift)
            .count_descending()
        )
        IF best_shifts[0].Shift NOT EQUALS target_shift:
            APPEND "Recommendation 2: Historical data indicates optimal performance on [" + best_shifts[0].Shift + "] Shift for this product variant." TO recommendations

    IF predicted_downtime > 20.0:
        APPEND "Operational Warning: High MTTR expected. Inject a standard buffer of " + round(predicted_downtime) + " minutes in the downstream assembly schedule." TO recommendations
        
    RETURN recommendations